In [2]:
# 1 - CAPTURA CANAIS
# Conjunto de funções que tem o objtivo de baixar dados de canais do youtube

from googleapiclient.errors import HttpError
from _controle_chaves_API import chave_api
from datetime import datetime

def busca_canais_por_handle(handle, maxResults=1, ordenador="relevance"):
    """
    Busca canais no YouTube por handle ou nome do canal e retorna informações básicas e estatísticas.

    :param handle (str): Pode ser uma arroba (@) ou o nome do canal.
    :param maxResults (int): Número máximo de canais a buscar.
    :param ordenador (str): Em qual ordem a API vai buscar os vídeos.
    :return: lista de dicionários com informações dos canais (com tipos corretos) ou None em caso de erro.
    """
    try:
        youtube = chave_api()
    except Exception as e:
        print(f"Erro ao autenticar a API: {e}")
        return None

    try:
        resposta = youtube.search().list(
            part="snippet",
            q=handle,
            type="channel",
            maxResults=maxResults,
            order=ordenador
        ).execute()
    except HttpError as e:
        print(f"[YouTube API] Erro HTTP: {e.resp.status} - {getattr(e, 'error_details', e)}")
        return None
    except Exception as e:
        print(f"[Erro inesperado] {type(e).__name__}: {e}")
        return None

    if not resposta.get("items"):
        print(f"Nenhum canal encontrado para: {handle}")
        return []

    canais = []

    def to_int_safe(val):
        try:
            return int(val)
        except:
            return 0

    for item in resposta["items"]:
        if item['id']['kind'] == 'youtube#channel':
            canal_id = item['id']['channelId']

            # Valores básicos
            titulo = item['snippet']['title']
            descricao = item['snippet']['description']
            data_raw = item['snippet']['publishedAt']

            try:
                data_criacao = datetime.fromisoformat(data_raw.replace("Z", "+00:00")).date()
            except Exception:
                data_criacao = None  # Em caso de erro, pode usar None

            # Valores default
            seguidores = views = n_videos = 0
            autor = "N/A"

            # Busca estatísticas
            try:
                resposta_stats = youtube.channels().list(
                    part="snippet,statistics",
                    id=canal_id
                ).execute()
                dados = resposta_stats['items'][0]

                stats = dados.get("statistics", {})
                snippet = dados.get("snippet", {})

                seguidores = to_int_safe(stats.get("subscriberCount"))
                views = to_int_safe(stats.get("viewCount"))
                n_videos = to_int_safe(stats.get("videoCount"))
                autor = snippet.get("customUrl") or snippet.get("handle") or "N/A"

            except Exception as e:
                print(f"[Erro ao buscar estatísticas para canal {canal_id}] {type(e).__name__}: {e}")

            canais.append({
                "id_ch": canal_id,               # str
                "titulo": titulo,                # str
                "descricao": descricao,          # str
                "data_criacao": data_criacao,    # datetime.date
                "seguidores": seguidores,        # int
                "views": views,                  # int
                "n_videos": n_videos,            # int
                "autor": autor                   # str
            })

    return canais

def busca_canal_por_id(id_cn):
    """
    Busca informações de um canal do YouTube a partir do seu ID.

    :param id_cn (str): ID do canal no YouTube (ex.: 'UC_x5XG1OV2P6uZZ5FSM9Ttw').
    :return: Uma lista com um dicionário contendo as informações do canal com tipos corretos, ou None em caso de erro.
    """
    try:
        youtube = chave_api()
    except Exception as e:
        print(f"Erro ao autenticar a API: {e}")
        return None

    try:
        resposta = youtube.channels().list(
            part="snippet,statistics",
            id=id_cn
        ).execute()
    except HttpError as e:
        print(f"[YouTube API] Erro HTTP: {e.resp.status} - {getattr(e, 'error_details', e)}")
        return None
    except Exception as e:
        print(f"[Erro inesperado] {type(e).__name__}: {e}")
        return None

    if not resposta.get("items"):
        print(f"Nenhum canal encontrado para o ID: {id_cn}")
        return None

    def to_int_safe(val):
        try:
            return int(val)
        except:
            return 0

    try:
        item = resposta["items"][0]
        snippet = item.get("snippet", {})
        stats = item.get("statistics", {})

        data_raw = snippet.get("publishedAt", "")
        try:
            data_criacao = datetime.fromisoformat(data_raw.replace("Z", "+00:00")).date()
        except Exception:
            data_criacao = None

        canal = {
            "id_ch": item.get("id", ""),                            # str
            "titulo": snippet.get("title", ""),                     # str
            "descricao": snippet.get("description", ""),           # str
            "data_criacao": data_criacao,                           # datetime.date
            "autor": snippet.get("customUrl") or snippet.get("handle") or "N/A",  # str
            "seguidores": to_int_safe(stats.get("subscriberCount")),  # int
            "views": to_int_safe(stats.get("viewCount")),             # int
            "n_videos": to_int_safe(stats.get("videoCount"))          # int
        }

        return [canal]

    except Exception as e:
        print(f"[Erro ao processar os dados do canal] {type(e).__name__}: {e}")
        return None

Sucesso com a chave: key_nathalia
Pulando para a próxima chave API...


In [4]:
canal_nikolas = busca_canais_por_handle("Nikolas Ferreira", 1, "relevance")

Sucesso com a chave: key_sandra2
Pulando para a próxima chave API...


In [6]:
# 2 - CAPTURA VIDEOS
# Conjunto de funções que tem o objetivos de baixar N vídeos de um canal do youtube
from datetime import datetime
import time
from _controle_chaves_API import chave_api

def get_uploads_playlist_id(youtube, channel_id):
    """Retorna o ID da playlist de uploads do canal"""
    resposta = youtube.channels().list(
        part="contentDetails",
        id=channel_id
    ).execute()

    items = resposta.get("items", [])
    if not items:
        raise Exception("Canal não encontrado ou sem vídeos.")

    return items[0]["contentDetails"]["relatedPlaylists"]["uploads"]

def get_video_ids_from_playlist(youtube, playlist_id, max_videos=50):
    """Coleta todos os IDs dos vídeos da playlist"""
    video_ids = []
    next_page_token = None

    while True:
        resposta = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()

        for item in resposta["items"]:
            video_ids.append(item["contentDetails"]["videoId"])
            if len(video_ids) >= max_videos:
                return video_ids

        next_page_token = resposta.get("nextPageToken")
        if not next_page_token:
            break

    return video_ids

def get_video_details(youtube, video_ids):
    """Pega os detalhes dos vídeos a partir dos IDs, convertendo tipos"""
    dados = []

    for i in range(0, len(video_ids), 50):
        ids = video_ids[i:i + 50]
        resposta = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(ids)
        ).execute()

        for item in resposta["items"]:
            snippet = item.get("snippet", {})
            statistics = item.get("statistics", {})

            data_criacao_iso = snippet.get("publishedAt")
            data_criacao = datetime.strptime(data_criacao_iso, "%Y-%m-%dT%H:%M:%SZ").date() if data_criacao_iso else None

            views = int(statistics.get("viewCount", 0))
            likes = int(statistics.get("likeCount", 0))
            coments = int(statistics.get("commentCount", 0))

            dados.append({
                "id_vd": item["id"],
                "id_ch": snippet.get("channelId", ""),
                "titulo": snippet.get("title", ""),
                "descricao": snippet.get("description", ""),
                "data_criacao": data_criacao,
                "views": views,
                "likes": likes,
                "coments": coments,
            })

        time.sleep(0.5)  # Para evitar limites de cota

    return dados

def busca_videos_de_um_canal(channel_id, limite=50):
    youtube = chave_api()
    playlist_id = get_uploads_playlist_id(youtube, channel_id)
    video_ids = get_video_ids_from_playlist(youtube, playlist_id, max_videos=limite)
    dados = get_video_details(youtube, video_ids)

    return dados


In [7]:
# Baixa os vídeos
videos_nikolas = busca_videos_de_um_canal('UCxI9vN6UbxmBt8VIvUKtJaA',500)

Sucesso com a chave: key_xaxin
Pulando para a próxima chave API...


In [11]:
# Salva os vídeos em json
import json


# converte o campo 'data_criacao' de cada dicionário na lista
# para uma string. O json não sabe tratar o tipo datetime :(
for video in videos_nikolas:
    if 'data_criacao' in video:
        video['data_criacao'] = video['data_criacao'].isoformat() # Transforma em "AAAA-MM-DD"


CAMINHO_ARQUIVO = "C:\\Users\\bertolo\\tese_nikolas_ferreira\\corpora\\videos_nikolas.json"

with open(CAMINHO_ARQUIVO, "w", encoding="utf-8") as arquivo:
    json.dump(videos_nikolas, arquivo, indent=4, ensure_ascii=False)


In [8]:
# Soma da audiência dos vídeos
audiencia = 0

for v in videos_nikolas:
    audiencia += v.get("views")